# 00. Quick start

Forecast Peyton Manning's Wikipedia page views in one call, then read what comes back: a short summary, the forecast table, the forecast chart, the components chart and a one-page report. By the end you will know which method to reach for next, and where each later notebook in the series picks up.

- Load the data
- Forecast in one call
- Read the result
- Chart the forecast
- Chart the components
- Peek at the one-page report
- The rest of the series

Next: [01. Your data](01_your_data.ipynb)

In [ ]:
from prophet import forecast

from helpers import load_example, quiet_logs

quiet_logs()

## Load the data

NeoProphet needs a date column and a number column. This file has them as `ds` and `y`, the names upstream Prophet uses, but [01. Your data](01_your_data.ipynb) shows that other names work too. Here `y` is the natural log of daily page views, so a `y` of 9 means about 8,100 views.

In [ ]:
df = load_example("peyton_manning")
print(f"{len(df):,} rows, {df['ds'].min():%d %b %Y} to {df['ds'].max():%d %b %Y}")
df.head()

## Forecast in one call

`forecast()` finds the columns, works out the spacing between observations, fits the model and predicts the horizon. A horizon like `"12 months"` is turned into a number of daily steps for you, and `country_holidays="US"` adds the US public holidays as effects the model can learn.

The series is missing 59 days, so `freq="D"` tells NeoProphet that daily spacing is intended. Without it the forecast is the same, but a log line points out the gaps (see [01. Your data](01_your_data.ipynb)).

In [ ]:
result = forecast(df, horizon="12 months", country_holidays="US", freq="D")
print(result.summary())

## Read the result

`result.forecast` holds one row per future day: `yhat` is the central forecast, and `yhat_lower` and `yhat_upper` bound an 80% uncertainty interval by default. `result.model` is the fitted `Prophet` model, `result.history` the cleaned training data and `result.predictions` the full table including every component, which is what the charts below draw from.

In [ ]:
result.forecast.tail()

## Chart the forecast

The title and subtitle are written from the data, dots are the observations, the shaded zone starts where the data ends and the callouts mark the largest changes in trend. The interval covers uncertainty in the trend and the noise, but not every kind of surprise, so treat it as a guide and not a guarantee ([05. Uncertainty and outliers](05_uncertainty_and_outliers.ipynb) measures how often it holds).

In [ ]:
fig = result.plot()

Takeaway: the model expects the winter peaks to return and the underlying level to keep drifting down. Because `y` is a log here, the "4.4% lower" in the subtitle is a change in log views, not in views.

## Chart the components

The forecast is a sum of parts: a trend, repeating patterns and holiday effects. Seeing them apart is the quickest way to find out why the model forecasts what it does.

In [ ]:
fig = result.plot_components()

Takeaway: the trend has slipped about 6% since 2007, Mondays are the busiest day, January the busiest month and Christmas Day the deepest dip (-1.08 in log units, so about two thirds fewer views than usual).

## Peek at the one-page report

`result.report()` puts the forecast, key numbers, components and a backtest on a single A4 page. The backtest holds out the last stretch of the history, refits on the rest and scores that forecast against what happened, so this call takes a few seconds longer than the charts above. Saving it as a PDF is one more line, `fig.savefig("report.pdf")`; [07. Reports and dashboards](07_reports_and_dashboards.ipynb) covers the report and the interactive dashboard.

In [ ]:
fig = result.report()

Takeaway: on the held-back year the typical error was about 5% and roughly 80% of the real values fell inside the 80% band. That is a good result for this series, not a promise for yours, so run the backtest on your own data before relying on a forecast.

## The rest of the series

Each notebook takes one question further than this one and runs in a few minutes. Start with 01 if your data is not already a tidy `ds` and `y` table; otherwise follow your own question.

| Notebook | What it covers |
|---|---|
| [00. Quick start](00_quick_start.ipynb) | This notebook: one call, the summary, the charts and the report |
| [01. Your data](01_your_data.ipynb) | Accepted input shapes, bad-data errors, frequency and horizons, closed days and hours |
| [02. Trend](02_trend.ipynb) | How the trend bends, how flexible it should be and what that does to the forecast |
| [03. Seasonality and holidays](03_seasonality_and_holidays.ipynb) | Yearly, weekly and daily patterns, additive against multiplicative, holidays and special events |
| [04. Regressors and weather](04_regressors_and_weather.ipynb) | Extra columns that explain the series, local temperature, and the regressor effects chart |
| [05. Uncertainty and outliers](05_uncertainty_and_outliers.ipynb) | What the intervals mean, how often they cover, and how outliers distort a fit |
| [06. Evaluation and tuning](06_evaluation_and_tuning.ipynb) | Backtests, cross-validation, metrics and `tune=True`, including when tuning does not help |
| [07. Reports and dashboards](07_reports_and_dashboards.ipynb) | The one-page PDF report and the offline HTML dashboard |
| [08. Many series on Apple Silicon](08_many_series_mlx.ipynb) | Fitting many series at once with the experimental Apple Silicon backend |
| [09. Benchmark against upstream](09_benchmark_vs_upstream.ipynb) | Accuracy and speed against upstream Prophet |

## Next

Continue with [01. Your data](01_your_data.ipynb): the shapes of data `forecast()` accepts, what it says when the data is wrong, and how it handles different frequencies and closed days.